# TF-QKD optimization for each state
This notebook imports the rewritten Python functions and optimizes VSS and GS states.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tfqkd.key_rate import tf_qkd_fast
from tfqkd.optimization import optimize_vss, optimize_gs

## Basic sanity checks

In [ ]:
N_truncation = 6
pd = 1e-8
loss_db = 0.0

vss_parameters = [0.9, 0.0, np.pi]
gs_parameters = [0.0, 0.3, 0.0, np.pi, 0.1, 0.0, np.pi]

print('VSS:', tf_qkd_fast(loss_db, N_truncation, vss_parameters, 'VSS', pd))
print('GS :', tf_qkd_fast(loss_db, N_truncation, gs_parameters, 'GS', pd))

## Optimize VSS for one loss value

In [ ]:
vss_result = optimize_vss(loss_db=0.0, n_truncation=N_truncation, pd=pd, seed=1, polish=True)
vss_result

## Optimize GS for one loss value

In [ ]:
gs_result = optimize_gs(loss_db=0.0, n_truncation=N_truncation, pd=pd, seed=1, polish=True)
gs_result

## Sweep loss values
Start with a small sweep. Increase the number of loss points after verification.

In [ ]:
loss_values = np.linspace(0, 30, 7)
rows = []

for loss in loss_values:
    out = optimize_vss(loss_db=float(loss), n_truncation=N_truncation, pd=pd, seed=1, polish=False)
    rows.append({
        'state': out['state'],
        'loss_db': loss,
        'rate': out['rate'],
        'e_x': out['e_x'],
        'e_z': out['e_z'],
        'p_xx': out['p_xx'],
        'x': out['x'],
    })

df = pd.DataFrame(rows)
df

In [ ]:
plt.figure(figsize=(6,4))
plt.semilogy(df['loss_db'], df['rate'], marker='o')
plt.xlabel('Loss (dB)')
plt.ylabel('Optimized key rate')
plt.grid(True, which='both')
plt.show()

## Save results

In [ ]:
output_path = Path('optimized_vss_results.csv')
df.to_csv(output_path, index=False)
output_path